## Step 1: Install Python, Create a Virtual Environment

This course requires **Python 3.12**. Avoid Python 3.14+ (it has compatibility issues with the `openai` package).

### Install Python

**macOS:**
```bash
brew install python@3.12
```
Or download the installer from [python.org/downloads](https://www.python.org/downloads/).

**Windows:**
```powershell
winget install -e --id Python.Python.3.12
```
Or download from [python.org/downloads](https://www.python.org/downloads/). During installation, check **"Add python.exe to PATH"**.

**Linux (Ubuntu/Debian):**
```bash
sudo apt update && sudo apt install python3.12 python3.12-venv
```

> **Verify:** Run `python3 --version` (macOS/Linux) or `py -3 --version` (Windows) — you should see `3.12.x`.

### Create a Virtual Environment

```bash
cd /path/to/MicrosoftFoundry
python3 -m venv .venv

# Activate:
source .venv/bin/activate          # macOS / Linux
.venv\Scripts\activate             # Windows
```

> **Known issue:** The `pip` command in this venv may have a broken shebang. Use `python -m pip install ...` instead of `pip install ...` when working in the terminal. The `%pip` magic in notebooks works fine.

# Developer Environment Setup

This notebook sets up your environment for the entire **Azure AI Foundry** course. Complete it once before starting any section.


## Prerequisites

You need these before continuing:

1. **Azure account** with an active subscription 
2. **Azure AI Foundry project** with a deployed model (e.g., `gpt-4.1-mini`) 
3. **Azure AI User** role (minimum) on your Foundry project — if you created the project, you already have sufficient permissions
4. **VS Code** installed — [download here](https://code.visualstudio.com/Download)


In [ ]:
import sys
print(f"Python: {sys.version}")
print(f"Path:   {sys.executable}")

major, minor = sys.version_info[:2]
if ".venv" not in sys.executable:
    print("\n⚠️  Not running from .venv — select the correct kernel (see Step 2)")
elif minor > 13:
    print(f"\n⚠️  Python 3.{minor} detected — 3.12 is recommended (3.14+ has compatibility issues)")
elif minor >= 10:
    print("\n✅ Python version and venv look good.")
else:
    print("\n❌ Python 3.10+ required. Install Python 3.12.")

## Step 2: Jupyter Kernel

The **kernel** is the Python interpreter that runs your notebook. You want it pointing at your `.venv`.

**To select the kernel in VS Code:**
1. Click the kernel name in the **top-right corner** of any notebook
2. Choose **Python Environments** → select the `.venv` Python

**If `.venv` doesn't appear:**
1. `Cmd+Shift+P` (macOS) or `Ctrl+Shift+P` (Windows/Linux)
2. Type **Python: Select Interpreter** → **Enter interpreter path** → navigate to `.venv/bin/python`

**Optional — create a named kernel:**
```bash
source .venv/bin/activate
python -m pip install ipykernel
python -m ipykernel install --user --name=foundry
```

## Step 3: VS Code Extensions

**Required:**
- [Python](https://marketplace.visualstudio.com/items?itemName=ms-python.python)
- [Jupyter](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter)
- [Foundry](https://marketplace.visualstudio.com/items?itemName=TeamsDevApp.vscode-ai-foundry)



## Step 4: Azure CLI

All sections authenticate via `DefaultAzureCredential`, which picks up your `az login` session.

**Install:**
```bash
brew update && brew install azure-cli    # macOS
winget install -e --id Microsoft.AzureCLI  # Windows
curl -sL https://aka.ms/InstallAzureCLIDeb | sudo bash  # Linux
```

**Sign in:**
```bash
az login                    # Opens browser
az login --use-device-code  # For remote/SSH environments
```

In [ ]:
import subprocess, json

result = subprocess.run(["az", "account", "show"], capture_output=True, text=True)
if result.returncode == 0:
    acct = json.loads(result.stdout)
    print(f"✅ Signed in as {acct['user']['name']}")
    print(f"   Subscription: {acct['name']}")
else:
    print("❌ Not signed in. Run 'az login' in your terminal first.")

## Step 5: Core Packages

These packages are used across most sections. Each section's notebooks also have their own `%pip install` cell, so you can run any notebook independently.

| Package | Purpose |
|---|---|
| `azure-ai-projects==2.0.0b3` | Foundry SDK (agents, workflows, conversations). **Must be b3** — b2 is incompatible with agent-framework. |
| `azure-identity` | Authentication via `DefaultAzureCredential` |
| `python-dotenv` | Load `.env` files |
| `ipykernel` | Create Jupyter kernels from `.venv` |

Section-specific packages (e.g., `agent-framework`, `azure-ai-contentsafety`, `pydantic`) are installed within each section's notebooks.

In [ ]:
%pip install azure-ai-projects==2.0.0b3 azure-identity python-dotenv ipykernel

In [ ]:
try:
    import azure.ai.projects
    print(f"✅ azure-ai-projects {azure.ai.projects.__version__}")
except ImportError:
    print("❌ azure-ai-projects not installed")

try:
    import azure.identity
    print("✅ azure-identity installed")
except ImportError:
    print("❌ azure-identity not installed")

try:
    import dotenv
    print("✅ python-dotenv installed")
except ImportError:
    print("❌ python-dotenv not installed")

## Step 6: Environment Variables

Each course section has its own `.env` file with configuration specific to that section. Most sections needs these two variables:

```
FOUNDRY_PROJECT_ENDPOINT="https://YOUR-RESOURCE.services.ai.azure.com/api/projects/YOUR-PROJECT"
MODEL_DEPLOYMENT_NAME="gpt-4.1-mini"
```

Some sections add extra variables. For example:


**Find your endpoint:** [ai.azure.com](https://ai.azure.com) → your project → **Overview** page → copy **Foundry project endpoint**

### How `.env` loading works

`load_dotenv()` uses `find_dotenv()` internally, which searches **upward** from the current notebook's directory. 

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model = os.getenv("MODEL_DEPLOYMENT_NAME")

if endpoint:
    print(f"✅ FOUNDRY_PROJECT_ENDPOINT: {endpoint}")
else:
    print("❌ FOUNDRY_PROJECT_ENDPOINT not set — create a .env file (see above)")

if model:
    print(f"✅ MODEL_DEPLOYMENT_NAME: {model}")
else:
    print("❌ MODEL_DEPLOYMENT_NAME not set")

In [ ]:
# My Addition: use absolute path to load .env, and print the values to confirm they are loaded correctly.
# Load the .env file — this must run before we read any values: NOTE: I use an absolute path here to avoid confusion about where the .env file should be located.
load_dotenv("/Users/macbookpro/dev/python/github/python/foundry-nett/00-Setup/.env")

# Read each setting from the environment
my_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")       # Foundry project URL (for testing later)
subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")       # Azure subscription ID
resource_group = os.getenv("AZURE_RESOURCE_GROUP")         # Resource group name
account_name = os.getenv("FOUNDRY_ACCOUNT_NAME")           # Cognitive Services account name

# Quick check — make sure nothing is missing
print("Endpoint:", my_endpoint)
print("Subscription:", subscription_id)
print("Resource group:", resource_group)
print("Account name:", account_name)

## Step 7: Smoke Test

This connects to your Foundry project and sends a test message. If it works, your environment is ready.

> **Before running:** Replace `YOUR-ENDPOINT` and `YOUR-MODEL` in the cell below with the values from your Foundry project. You can find them on the project **Overview** page at [ai.azure.com](https://ai.azure.com).

In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

load_dotenv(override=True)

# ✏️ Insert your own values:
project_url = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")  

foundry_client = AIProjectClient(
    endpoint=project_url,
    credential=DefaultAzureCredential(),
)
oai = foundry_client.get_openai_client()

response = oai.responses.create(
    model=deployment_name,
    input="Say 'Hello from Microsoft Foundry!' and nothing else.",
)

print(f"✅ {response.output_text}")
print("\nYour environment is ready. You can start with the demos!")